Task0：前置設定與載入套件

In [ ]:
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 初始化 GEE (若出現驗證錯誤，請加上 ee.Authenticate() 重新驗證)
ee.Initialize(project='gis-gee2026') 

# 定義太魯閣/秀林研究區 BBOX
TAROKO_BBOX = [121.34526379253053, 24.046021742135874, 121.85149217685861, 24.35767637905926]
aoi = ee.Geometry.Rectangle(TAROKO_BBOX)

print(f"Study area: Xiulin / Taroko")
print(f"BBOX: {TAROKO_BBOX}")
print(f"Time range: 2000–2026 (26 years)")

步驟一：Task 1 - Landsat 波段調和與 26 年 NDVI 時序 (25%)

In [ ]:
# ==========================================
# Task 1: Landsat Harmonization & 26-Year NDVI
# ==========================================

# 1. 定義波段調和函數 (將 L5/L7 與 L8/L9 統一命名)
def harmonize_l57(image):
    return (image.select(
        ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'QA_PIXEL'],
        ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'QA_PIXEL']
    ).copyProperties(image, ['system:time_start']))

def harmonize_l89(image):
    return (image.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'QA_PIXEL'],
        ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'QA_PIXEL']
    ).copyProperties(image, ['system:time_start']))

# 2. 定義雲遮罩與數值校正函數
def apply_scale_and_mask(image):
    qa = image.select('QA_PIXEL')
    cloud = qa.bitwiseAnd(1 << 3).eq(0)
    shadow = qa.bitwiseAnd(1 << 4).eq(0)
    mask = cloud.And(shadow)

    spectral = (image.select(['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2'])
        .multiply(0.0000275).add(-0.2)
        .clamp(0, 1))
    return spectral.updateMask(mask).copyProperties(image, ['system:time_start'])

# 3. 讀取並合併四代 Landsat 影像 (2000-2026)
l5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').filterBounds(aoi).filterDate('2000-01-01', '2012-12-31').map(harmonize_l57)
l7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2').filterBounds(aoi).filterDate('2000-01-01', '2026-12-31').map(harmonize_l57)
l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(aoi).filterDate('2013-01-01', '2026-12-31').map(harmonize_l89)
l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterBounds(aoi).filterDate('2021-01-01', '2026-12-31').map(harmonize_l89)

landsat_all = l5.merge(l7).merge(l8).merge(l9).map(apply_scale_and_mask)
print(f"Total Landsat images (2000–2026): {landsat_all.size().getInfo()}")

# 4. 計算 NDVI
def compute_ndvi(image):
    ndvi = image.normalizedDifference(['NIR', 'Red']).rename('NDVI')
    return ndvi.copyProperties(image, ['system:time_start'])

ndvi_collection = landsat_all.map(compute_ndvi)

# 5. 萃取 26 年的年度平均 NDVI 數值 (這段需要向 Google 請求資料，約需 10-30 秒)
print("正在計算 26 年年度 NDVI 中位數，請稍候...")
years = list(range(2000, 2027))
annual_ndvi_data = []

for year in years:
    yearly_img = ndvi_collection.filterDate(f'{year}-01-01', f'{year}-12-31').median()
    # 檢查該年度是否有資料
    mean_dict = yearly_img.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=30, maxPixels=1e9)
    ndvi_val = mean_dict.get('NDVI').getInfo()
    annual_ndvi_data.append(ndvi_val)

# 6. 繪製趨勢圖
plt.figure(figsize=(12, 5))
plt.plot(years, annual_ndvi_data, marker='o', linestyle='-', color='forestgreen', label='Annual Mean NDVI')

# 加入 2024 地震標記
plt.axvline(x=2024, color='red', linestyle='--', linewidth=2, label='2024 Earthquake')

# 計算並繪製線性趨勢線
valid_data = [(y, v) for y, v in zip(years, annual_ndvi_data) if v is not None]
x_val = np.array([d[0] for d in valid_data])
y_val = np.array([d[1] for d in valid_data])
m, b = np.polyfit(x_val, y_val, 1)
plt.plot(x_val, m*x_val + b, color='orange', linestyle='-', linewidth=2, label=f'Trendline (slope: {m:.5f})')

plt.title('26-Year NDVI Trend in Xiulin / Taroko (2000-2026)')
plt.xlabel('Year')
plt.ylabel('Mean NDVI')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

Task 1 Analysis:
從 26 年的 NDVI 趨勢圖可看出，該區域長期呈現微幅的綠化趨勢 (Greening)，線性趨勢線斜率為 0.00279。在 2000 至 2023 年間，平均 NDVI 數值多穩定維持在 0.60 至 0.70 之間波動。然而，2024 年的數值出現顯著的異常低谷，陡降至約 0.55，對應了 0403 地震造成的巨量植被破壞。相比歷史上其他擾動年份，2024 年的下降幅度最為劇烈，凸顯此次地震在過去 26 年觀測紀錄中的極端性。

In [ ]:
# ==========================================
# Task 2: 像素級線性趨勢分析 (Pixel-Level Trend)
# ==========================================
print("準備進行像素級線性趨勢分析與統計 (需時約 10-20 秒)...")

# 1. 建立帶有時間波段 (Time Band) 的年度 NDVI 影像集合
def annual_ndvi_image(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)
    median_ndvi = ndvi_collection.filterDate(start, end).median()
    # 新增一個名稱為 'time' 的波段，數值為該年份 (float)
    time_band = ee.Image.constant(year).float().rename('time')
    return median_ndvi.addBands(time_band).set('system:time_start', start.millis())

year_list = ee.List.sequence(2000, 2026)
annual_col = ee.ImageCollection(year_list.map(lambda y: annual_ndvi_image(ee.Number(y).int())))

# 2. 執行線性迴歸 (linearFit: 依變數 NDVI, 自變數 time)
# 'scale' 波段即為斜率 (Slope)，'offset' 為截距
trend = annual_col.select(['time', 'NDVI']).reduce(ee.Reducer.linearFit())
slope = trend.select('scale')

# 3. 統計綠化、退化與穩定的比例 (設定閾值為 +/- 0.001)
greening = slope.gt(0.001)
browning = slope.lt(-0.001)
stable = slope.gte(-0.001).And(slope.lte(0.001))

# 安全的 reduceRegion 求和函數，避免不同輸出鍵名造成錯誤
def safe_region_sum(image, geometry, scale):
    result = image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geometry,
        scale=scale,
        maxPixels=1e9,
        bestEffort=True,
        tileScale=4
    ).getInfo()
    if not result:
        return 0
    return list(result.values())[0] or 0

# 為了加快雲端計算速度，這裡用 100m 尺度來概算面積比例
green_area = safe_region_sum(greening.multiply(ee.Image.pixelArea()), aoi, 100)
brown_area = safe_region_sum(browning.multiply(ee.Image.pixelArea()), aoi, 100)
stable_area = safe_region_sum(stable.multiply(ee.Image.pixelArea()), aoi, 100)

total_area = green_area + brown_area + stable_area

print("\n--- 26 年 NDVI 空間趨勢統計 ---")
print(f"🟢 長期綠化 (Greening, slope > 0.001): {green_area/total_area*100:.1f}%")
print(f"🔴 長期退化 (Browning, slope < -0.001): {brown_area/total_area*100:.1f}%")
print(f"⚪ 穩定狀態 (Stable)                  : {stable_area/total_area*100:.1f}%")

# 4. 繪製互動式地圖
Map_Trend = geemap.Map(center=[24.2, 121.6], zoom=11)
# 視覺化參數：紅 (退化/負斜率) -> 白 (穩定) -> 綠 (綠化/正斜率)
trend_vis = {'min': -0.005, 'max': 0.005, 'palette': ['red', 'white', 'green']}
Map_Trend.addLayer(slope.clip(aoi), trend_vis, '26-Year NDVI Trend (Slope)')
Map_Trend.addLayerControl()

# 5. 將斜率圖匯出至 Google Drive (作業要求之一)
task_trend = ee.batch.Export.image.toDrive(
    image=slope,
    description='taroko_ndvi_trend_26yr',
    folder='GEE_Exports',
    region=aoi,
    scale=30,
    crs='EPSG:32651',
    maxPixels=1e10
)
task_trend.start()
print("✅ GeoTIFF 匯出任務已送至背景執行 (稍後請至 Drive 檢查截圖)！")

# 顯示地圖
Map_Trend

從地圖觀察，研究區絕大部分的山林坡地在過去 26 年間呈現穩定或長期綠化（綠色與白色區塊）。長期退化（紅色區塊）呈現高度空間集中性，主要分布於兩大特徵區域：一是蜿蜒的河川廊道（受河流自然沖刷、改道與砂石堆積影響），二是圖面右下方的平緩地帶與人類活動區（反映都市擴張與土地開發）。

對比 W13 僅使用 6 年 (2020-2026) Sentinel-2 資料所呈現的「大面積嚴重退化」（主要捕捉到 0403 地震的短期強烈衝擊），W14 的 26 年 Landsat 長時序分析提供了更宏觀的歷史基線。這證明較長的觀測窗口能有效分離「短期極端擾動」與「長期環境趨勢」。在 26 年的尺度下，太魯閣多數山區植被展現了穩健的自我修復能力，整體維持在綠化的演替軌跡上；而真正的長期地貌劣化，多源自河川動態營力與人為長期的土地利用改變。

Task 3：桃園埤塘消失 MNDWI 偵測

In [ ]:
# ==========================================
# Task 3: 桃園埤塘消失 MNDWI 偵測
# ==========================================
import os
import json
import gdown

print("開始處理桃園埤塘 MNDWI 分析 (需時約 30-60 秒)...")

# 1. 定義桃園研究區
TAOYUAN_BBOX = [120.94, 24.83, 121.35, 25.08]
aoi_taoyuan = ee.Geometry.Rectangle(TAOYUAN_BBOX)

# 載入桃園區域的 Landsat 影像 (沿用 Task 1 的調和與校正函數)
l5_ty = (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
    .filterBounds(aoi_taoyuan)
    .filterDate('2000-01-01', '2012-12-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 40))
    .map(harmonize_l57))
l7_ty = (ee.ImageCollection('LANDSAT/LE07/C02/T1_L2')
    .filterBounds(aoi_taoyuan)
    .filterDate('2000-01-01', '2026-12-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 40))
    .map(harmonize_l57))
l8_ty = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(aoi_taoyuan)
    .filterDate('2013-01-01', '2026-12-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 40))
    .map(harmonize_l89))
l9_ty = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
    .filterBounds(aoi_taoyuan)
    .filterDate('2021-01-01', '2026-12-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 40))
    .map(harmonize_l89))

landsat_taoyuan = l5_ty.merge(l7_ty).merge(l8_ty).merge(l9_ty).map(apply_scale_and_mask)

# 2. 計算 MNDWI 與年度水體頻率
def compute_mndwi(image):
    mndwi = image.normalizedDifference(['Green', 'SWIR1']).rename('MNDWI')
    return mndwi.copyProperties(image, ['system:time_start'])

mndwi_taoyuan = landsat_taoyuan.select(['Green', 'SWIR1']).map(compute_mndwi)

def yearly_water(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)
    median_mndwi = mndwi_taoyuan.filterDate(start, end).median()
    water = median_mndwi.gt(0.1).rename('water')
    return water.set('system:time_start', start.millis())

# 3. 比較早期 (2000-2005) 與近期 (2021-2026) 變化
early_water = mndwi_taoyuan.filterDate('2000-01-01', '2005-12-31').median().gt(0.1).rename('early_water')
recent_water = mndwi_taoyuan.filterDate('2021-01-01', '2026-12-31').median().gt(0.1).rename('recent_water')

# 4. 繪製顯示圖時改用早期/近期平均水體頻率，避免計算完整 26 年頻率過慢
lost_ponds = early_water.And(recent_water.Not())
new_water = recent_water.And(early_water.Not())

# 4. 估算消失與新增水體面積 (加入 tileScale 解決記憶體超載)
def estimate_area(mask_image, geometry, scale=60):
    result = mask_image.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geometry,
        scale=scale,
        maxPixels=1e10,
        bestEffort=True,
        tileScale=8
    ).getInfo()
    if not result:
        return 0
    return list(result.values())[0] or 0

lost_area_m2 = estimate_area(lost_ponds, aoi_taoyuan, 90)
new_area_m2 = estimate_area(new_water, aoi_taoyuan, 90)
net_change_m2 = new_area_m2 - lost_area_m2

lost_area_ha = lost_area_m2 / 10000
new_area_ha = new_area_m2 / 10000
net_change_ha = net_change_m2 / 10000

print(f"估算消失埤塘面積: {lost_area_ha:.2f} 公頃")
print(f"估算新增水體面積: {new_area_ha:.2f} 公頃")
print(f"淨變化面積 (new - lost): {net_change_ha:.2f} 公頃")

# 5. 驗證偵測率
if not os.path.exists('taoyuan_ponds_223.geojson'):
    print("下載埤塘驗證資料...")
    gdown.download('https://drive.google.com/uc?id=1qwrIIELIJXbrBL_oCBTcoE-aoWq1bdXw', 'taoyuan_ponds_223.geojson', quiet=False)

with open('taoyuan_ponds_223.geojson') as f:
    ponds_geojson = json.load(f)

pond_features = [ee.Feature(ee.Geometry.Point(feat['geometry']['coordinates'])) for feat in ponds_geojson['features']]
ponds_fc = ee.FeatureCollection(pond_features)

mndwi_at_ponds = recent_water.unmask(0).sampleRegions(collection=ponds_fc, scale=60)
detected = mndwi_at_ponds.filter(ee.Filter.eq('recent_water', 1)).size()
counts = ee.List([ponds_fc.size(), detected]).getInfo()

print(f"已知埤塘總數: {counts[0]}")
print(f"MNDWI 成功偵測數: {counts[1]}")
print(f"偵測率: {counts[1]/counts[0]*100:.1f}%")

# 6. 繪製地圖
Map_Pond = geemap.Map(center=[24.95, 121.2], zoom=12)
Map_Pond.addLayer(recent_water.selfMask().clip(aoi_taoyuan), {'palette': ['blue']}, 'Recent Water')
Map_Pond.addLayer(lost_ponds.selfMask().clip(aoi_taoyuan), {'palette': ['red']}, 'Lost Ponds')
Map_Pond.addLayer(new_water.selfMask().clip(aoi_taoyuan), {'palette': ['green']}, 'New Water')
Map_Pond.addLayerControl()
Map_Pond

本次分析 MNDWI 閾值法（> 0.1）的偵測率為 89.2%（成功偵測 199/223 口已知埤塘 註:這是初版結果，跑太慢了先改掉）。未被偵測到的部分埤塘，主要原因推測為 Landsat 空間解析度（30m）無法有效辨識面積過微小的水體。

由空間分布圖觀察，藍色區塊（長期穩定水體）多分布於沿海及部分內陸區域非都會區；而紅色區塊（消失埤塘）則明顯散布於內陸都會區與開發熱區（如大園虎形崙、桃園藝文、崙平工業區、中壢龍岡），反映出都市擴張與土地利用變更對水體景觀的壓縮。埤塘的大量消失直接減少了地表原有的滯洪與蓄水緩衝空間，將顯著降低都會區面對短延時強降雨等極端氣候時的防洪韌性。

Task 4 植被韌性指標計算與總結

In [ ]:
# ==========================================
# Task 4: 植被韌性指標計算與總結 (Vegetation Resilience)
# ==========================================
print("開始計算植被韌性指標 (Recovery Ratio)...")

# 1. 定義三個分析時期
baseline_ndvi = ndvi_collection.filterDate('2020-01-01', '2023-12-31').median()
impact_ndvi = ndvi_collection.filterDate('2024-01-01', '2024-12-31').median()
recovery_ndvi = ndvi_collection.filterDate('2025-01-01', '2026-12-31').median()

# 2. 計算恢復比率 (Recovery Ratio)
# 公式: (Recovery - Impact) / (Baseline - Impact)
numerator = recovery_ndvi.subtract(impact_ndvi)
denominator = baseline_ndvi.subtract(impact_ndvi)

# 遮罩掉未受重大干擾的區域 (設定 NDVI 下降小於 0.05 視為無顯著災損，避免分母趨近於零)
damage_mask = denominator.abs().gt(0.05)

recovery_ratio = (numerator.divide(denominator)
    .updateMask(damage_mask)
    .rename('recovery_ratio')
    .clamp(-1, 2))  # 將極端值截斷，方便視覺化與統計

# 3. 統計各韌性等級的面積比例
degrading = recovery_ratio.lt(0)
slow_recovery = recovery_ratio.gte(0).And(recovery_ratio.lt(0.5))
recovering = recovery_ratio.gte(0.5).And(recovery_ratio.lte(1.0))
exceeded = recovery_ratio.gt(1.0)

# 為了加快雲端計算速度，使用較低解析度與 tileScale 進行面積估算
STAT_SCALE = 250
area_image = ee.Image.pixelArea()
stats_image = ee.Image.cat([
    damage_mask.rename('damaged'),
    degrading.rename('degrading'),
    slow_recovery.rename('slow_recovery'),
    recovering.rename('recovering'),
    exceeded.rename('exceeded')
]).multiply(area_image).unmask(0)

stats = stats_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=STAT_SCALE,
    maxPixels=1e10,
    bestEffort=True,
    tileScale=4
).getInfo()

# 如果某些分類沒有返回值，則預設為 0
stats = {
    'damaged_sum': stats.get('damaged_sum', 0),
    'degrading_sum': stats.get('degrading_sum', 0),
    'slow_recovery_sum': stats.get('slow_recovery_sum', 0),
    'recovering_sum': stats.get('recovering_sum', 0),
    'exceeded_sum': stats.get('exceeded_sum', 0)
}

damaged_area = stats['damaged_sum']
deg_area = stats['degrading_sum']
slow_area = stats['slow_recovery_sum']
rec_area = stats['recovering_sum']
exc_area = stats['exceeded_sum']

if damaged_area and damaged_area > 0:
    print("\n--- 2024 地震受損區域之植被韌性恢復統計 ---")
    print(f"🔴 持續劣化 (Degrading, RR < 0)       : {deg_area/damaged_area*100:.1f}%")
    print(f"🟡 緩慢恢復 (Slow Recovery, 0 < RR < 0.5): {slow_area/damaged_area*100:.1f}%")
    print(f"🟢 穩定恢復 (Recovering, 0.5 < RR < 1.0) : {rec_area/damaged_area*100:.1f}%")
    print(f"🔵 超越基準 (Exceeded, RR > 1.0)       : {exc_area/damaged_area*100:.1f}%")

# 4. 繪製互動式地圖
Map_Resilience = geemap.Map(center=[24.2, 121.6], zoom=11)
# 視覺化參數設定: 紅(持續劣化) -> 黃(緩慢恢復) -> 綠(穩定恢復) -> 藍(超越基準)
resilience_vis = {
    'min': -0.2,
    'max': 1.2,
    'palette': ['red', 'yellow', 'green', 'blue']
}

Map_Resilience.addLayer(recovery_ratio.clip(aoi), resilience_vis, 'Vegetation Resilience (Recovery Ratio)')
Map_Resilience.addLayerControl()
Map_Resilience

Task 4: Integration Summary Report
1. 跨週連結與時空全貌 (Cross-week Connections)
遙測技術的演進構建了完整的 4D 觀測視角。W6 的 Kriging 空間內插法讓我們得以將點狀觀測（如雨量測站）轉換為二維連續表面；而 W14 的 GEE 時序分析則將時間軸延長至 26 年。空間的連續性結合時間的縱深，不僅能捕捉單次極端事件（如 0403 地震），更能精確標定環境基線與歷史脆弱度，這對於結合機器學習演算法建構高精度的災害潛勢圖（如期末專案的路網淹水脆弱度評估）是不可或缺的特徵變數。

2. W13 (6年) vs W14 (26年) 比較與應用時機
W13 (Sentinel-2, 10m) 提供了極高解析度的局部快照，適合用於災後立即的精細損害圈繪與關鍵交通節點中斷的空間定位。然而，6 年的數據容易將短期擾動誤判為永久性破壞。W14 (Landsat, 30m) 透過 26 年的長時序，成功過濾了短期的季節性雜訊，釐清了該區域長期綠化的背景趨勢。實務上，Landsat 適合用來評估歷史尺度的環境韌性，Sentinel-2 則適合微觀的工程災損辨識，兩者結合能同時達成宏觀潛勢分析與微觀搶修規劃。

3. 植被韌性評估 (Resilience Assessment)
由實際產出的 Recovery Ratio 地圖觀察，太魯閣受損區域的韌性呈現極度兩極化的空間分布。圖面中，紅色（持續劣化）與黃色（緩慢恢復）區塊高度密集地集中於「清水斷崖沿線」以及「立霧溪峽谷」兩側的陡峻邊坡。這印證了深層崩塌導致表土流失後，自然復育能力極低。相對地，圖面中南部（如三棧溪流域）與部分平緩坡地，則呈現大面積的綠色（穩定恢復）甚至藍色（超越基準），展現出該區山林強大的自然修復韌性。針對紅色低韌性熱區，因其緊鄰台9線（蘇花路廊）與台8線，應優先導入水保與邊坡防護工程。

4. 研究限制 (Limitations)
本次分析受限於 Landsat 30m 解析度，易產生混合像素效應（如圖中海岸線邊緣出現的藍色雜訊，多為水陸交界之光譜誤差），無法精確解析狹窄道路的細部崩塌。此外，Landsat 7 於 2003 年後的 SLC-off 條帶破圖問題，以及山區常年厚重的雲覆蓋率，皆會減少有效觀測次數。透過年度中位數合成 (Median Composite) 雖可減輕此問題，但仍可能平滑掉部分極端的災變特徵。

Bonus 1: Multi-Index Dashboard (多指標儀表板)

In [ ]:
# ==========================================
# Bonus 1: Multi-Index Dashboard (多指標儀表板)
# ==========================================
import matplotlib.pyplot as plt

print("開始計算 NDVI, MNDWI, NBR 多指標 26 年時序 (需時約 30-60 秒)...")

# 1. 定義多指標計算函數
def compute_indices(image):
    ndvi = image.normalizedDifference(['NIR', 'Red']).rename('NDVI')
    mndwi = image.normalizedDifference(['Green', 'SWIR1']).rename('MNDWI')
    nbr = image.normalizedDifference(['NIR', 'SWIR2']).rename('NBR')
    return ndvi.addBands(mndwi).addBands(nbr).copyProperties(image, ['system:time_start'])

multi_index_col = landsat_all.map(compute_indices)

# 2. 計算歷年平均值
years = list(range(2000, 2027))
results = []

for year in years:
    yearly_img = multi_index_col.filterDate(f'{year}-01-01', f'{year}-12-31').median()
    # 使用大 scale 概算以加快速度
    mean_dict = yearly_img.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=100, maxPixels=1e9, tileScale=4
    )
    
    try:
        results.append({
            'year': year,
            'NDVI': mean_dict.get('NDVI').getInfo(),
            'MNDWI': mean_dict.get('MNDWI').getInfo(),
            'NBR': mean_dict.get('NBR').getInfo()
        })
    except Exception as e:
        print(f"Year {year} failed: {e}")
        results.append({'year': year, 'NDVI': None, 'MNDWI': None, 'NBR': None})

# 過濾掉空值以便畫圖
valid_results = [r for r in results if r['NDVI'] is not None and r['MNDWI'] is not None and r['NBR'] is not None]
x_years = [r['year'] for r in valid_results]
y_ndvi = [r['NDVI'] for r in valid_results]
y_mndwi = [r['MNDWI'] for r in valid_results]
y_nbr = [r['NBR'] for r in valid_results]

# 3. 繪製三面板儀表板
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
indices_data = [
    ('NDVI (Vegetation)', y_ndvi, 'forestgreen'),
    ('MNDWI (Water)', y_mndwi, 'steelblue'),
    ('NBR (Burn/Bare)', y_nbr, 'darkorange')
]

for i, (title, data, color) in enumerate(indices_data):
    axes[i].plot(x_years, data, color=color, linewidth=2, marker='o', markersize=5)
    axes[i].axvline(x=2024, color='red', linestyle='--', alpha=0.7, label='2024 Earthquake')
    axes[i].set_ylabel(title.split(' ')[0], fontweight='bold')
    axes[i].set_title(title, loc='left')
    axes[i].grid(True, alpha=0.3)
    if i == 0:
        axes[i].legend(loc='upper right')

axes[-1].set_xlabel('Year', fontweight='bold')
fig.suptitle('Taroko Multi-Index Dashboard (2000–2026)', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()

# 儲存圖表
plt.savefig('multi_index_dashboard.png', dpi=150)
print("✅ 多指標儀表板已儲存為 multi_index_dashboard.png")
plt.show()

Bonus 1 Analysis:
從多指標儀表板可見，三個指標在 26 年的長時序中展現出不同的變化特徵，特別是在應對 2024 年地震的衝擊上：

NDVI (植被)：2000 至 2023 年間呈現長期的綠化上升趨勢。2024 年出現明顯低谷，反映植被因崩塌遭剝離；但 2025-2026 年數值迅速反彈，顯示太魯閣整體植被仍具備良好的初步恢復韌性。

MNDWI (水體)：全期數值皆為負值，且呈現長期緩步下降趨勢。在 2024 年地震時並未出現異常波動，反而維持在低點，這表示該區的環境變異主要來自實體地貌破壞與植被流失，而非水文環境的改變。

NBR (火燒/裸地)：對 2024 年地震擾動的反應最為劇烈。對比 NDVI 的短期下降，NBR 在 2024 年呈現「斷崖式」重挫（自 0.25 暴跌至約 0.13），且災後（2025-2026）數值持續在底部盤整。這證明 NBR 對於深層崩塌造成的「裸岩與碎石大量暴露」具備極高的光譜敏感度。

總結： NDVI 適合用來觀察植被長期的健康與復育軌跡；但在針對極端地質災害（如大地震引發的大規模崩塌）進行災區辨識時，NBR 能提供比 NDVI 更強烈、更具鑑別度的光譜訊號。

In [ ]:
# ==========================================
# Bonus 2: NDVI Time-Lapse Animation (26 年動畫 GIF)
# ==========================================
import io
import requests
from PIL import Image, ImageDraw, ImageFont
import imageio
import os

print("開始製作 26 年 NDVI 時序動畫 (需向伺服器請求 27 張影像，約需 1-2 分鐘，請耐心等候)...")

years = list(range(2000, 2027))
ndvi_palette = ['brown', 'yellow', 'green', 'darkgreen']
frames = []

for year in years:
    try:
        # 取得該年度的 NDVI 中位數影像
        yearly_ndvi = ndvi_collection.filterDate(f'{year}-01-01', f'{year}-12-31').median()
        
        # 設定縮圖參數
        vis_params = {
            'min': 0.0,
            'max': 0.8,
            'palette': ndvi_palette,
            'region': aoi,
            'dimensions': 512, # 控制解析度避免檔案過大
            'crs': 'EPSG:3857'
        }
        
        # 取得影像的下載網址
        thumb_url = yearly_ndvi.getThumbURL(vis_params)
        response = requests.get(thumb_url)
        img = Image.open(io.BytesIO(response.content)).convert('RGB')
        
        # 在影像上繪製文字標籤
        draw = ImageDraw.Draw(img)
        
        # 標記重大事件
        label = f" {year} "
        if year == 2001:
            label += " (Typhoon Toraji 桃芝颱風)"
        elif year == 2009:
            label += " (Typhoon Morakot 莫拉克颱風)"
        elif year == 2024:
            label += " (0403 Earthquake ❗)"
            
        # 簡單加上黑色陰影讓白字更明顯
        draw.text((16, 16), label, fill='black')
        draw.text((15, 15), label, fill='white')
        
        frames.append(img)
        print(f"  ✅ {year} 處理完成")
        
    except Exception as e:
        print(f"  ❌ {year} 處理失敗: {e}")
        # 如果某年失敗，拿上一年的圖來補幀，保持動畫連貫
        if frames:
            frames.append(frames[-1])

# 將所有畫格儲存為 GIF 動畫
output_gif = 'taroko_ndvi_26yr_timelapse.gif'
imageio.mimsave(output_gif, [np.array(f) for f in frames], duration=0.8, loop=0)

print(f"\n🎉 動畫製作完成！已儲存為 {output_gif}")
print("請在 VS Code 檔案總管中點擊該檔案預覽。")

透過 26 年的 NDVI 動畫 GIF，我們能直觀地感受到太魯閣生態系的動態演替。動畫中最具視覺震撼力的三個時刻為：

長期的動態平衡 (2000-2023)： 畫面整體維持濃郁的深綠色，但細看立霧溪峽谷與河道兩側，會發現每年都有微小的黃褐色斑塊在變動，這展現了高山峽谷地區因自然風化、微小落石與水文沖刷所維持的「動態平衡」。

極端氣候的短暫衝擊 (如 2009 莫拉克颱風)： 雖然在部分年份遭遇強烈颱風，但畫面中僅有局部坡地轉為黃褐色，且在接下來的 1-2 年內迅速被綠色覆蓋，顯示出山林對氣候事件的強大恢復韌性。

2024 地震的毀滅性剝離： 這是 26 年動畫中最劇烈的一幀。2024 年畫面右側（清水斷崖至峽谷口）瞬間爆發出大面積的紅褐色與黃色斑塊，如同生態系的「大出血」，與前 20 幾年的穩定綠化形成極度強烈的對比，深刻證明了本次地質災害的史無前例。

In [ ]:
# ==========================================
# Bonus 3: Landsat × Sentinel-2 Cross-Sensor Analysis
# ==========================================
from scipy.stats import linregress

print("開始執行 Landsat 與 Sentinel-2 跨感測器分析 (約需 1-2 分鐘)...")

# 定義面積計算輔助函數
# 傳入二值影像，回傳面積總和 (m2)
def get_area(mask_image, scale=30):
    result = mask_image.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=scale,
        maxPixels=1e10,
        bestEffort=True,
        tileScale=4
    ).getInfo()
    if not result:
        return 0
    return list(result.values())[0] or 0

# 1. 載入 Sentinel-2 L2A 資料並計算 NDVI (2019-2026)
def s2_ndvi_calc(img):
    scl = img.select('SCL')
    # 保留植被(4,5)與裸地，遮蔽雲(8,9,10)與陰影(3)
    mask = scl.neq(8).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(3))
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return ndvi.updateMask(mask).copyProperties(img, ['system:time_start'])

s2_col = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(aoi) \
    .filterDate('2019-01-01', '2026-12-31') \
    .map(s2_ndvi_calc)

# 2. 計算 2019-2026 重疊年份的年均 NDVI
overlap_years = list(range(2019, 2027))
l89_ndvi_vals = []
s2_ndvi_vals = []

for y in overlap_years:
    # Landsat
    l_img = ndvi_collection.filterDate(f'{y}-01-01', f'{y}-12-31').median()
    l_val = l_img.reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=100, maxPixels=1e9).getInfo()
    l89_ndvi_vals.append(list(l_val.values())[0] if l_val and list(l_val.values())[0] else None)
    
    # Sentinel-2
    s_img = s2_col.filterDate(f'{y}-01-01', f'{y}-12-31').median()
    s_val = s_img.reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=100, maxPixels=1e9).getInfo()
    s2_ndvi_vals.append(list(s_val.values())[0] if s_val and list(s_val.values())[0] else None)

# 清理空值
valid_data = [(l, s) for l, s in zip(l89_ndvi_vals, s2_ndvi_vals) if l is not None and s is not None]
x_l89 = np.array([d[0] for d in valid_data])
y_s2 = np.array([d[1] for d in valid_data])

# 3. 繪製散點圖與計算 R^2
slope, intercept, r_value, p_value, std_err = linregress(x_l89, y_s2)
r_squared = r_value ** 2

plt.figure(figsize=(6, 6))
plt.scatter(x_l89, y_s2, color='blue', alpha=0.7)
plt.plot(x_l89, slope * x_l89 + intercept, color='red', label=f'Trendline ($R^2$ = {r_squared:.3f})')
plt.plot([0.5, 0.8], [0.5, 0.8], color='gray', linestyle='--', label='1:1 Line')
plt.xlabel('Landsat NDVI (30m)')
plt.ylabel('Sentinel-2 NDVI (10m)')
plt.title('Cross-Sensor NDVI Comparison (2019-2026)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 4. 多解析度地震損害面積比較 (ΔNDVI < -0.15 視為嚴重受損)
print("\n--- 0403 地震損害面積多解析度比較 ---")
l_pre = ndvi_collection.filterDate('2023-01-01', '2023-12-31').median()
l_post = ndvi_collection.filterDate('2024-01-01', '2024-12-31').median()
l_delta = l_post.subtract(l_pre)
l_damage = l_delta.lt(-0.15)

s_pre = s2_col.filterDate('2023-01-01', '2023-12-31').median()
s_post = s2_col.filterDate('2024-01-01', '2024-12-31').median()
s_delta = s_post.subtract(s_pre)
s_damage = s_delta.lt(-0.15)

l_dmg_area = get_area(l_damage) / 10000  # 轉為公頃
s_dmg_area = get_area(s_damage) / 10000

print(f"Landsat (30m) 估算嚴重受損面積: {l_dmg_area:.2f} 公頃")
print(f"Sentinel-2 (10m) 估算嚴重受損面積: {s_dmg_area:.2f} 公頃")

Bonus 3 Analysis (Cross-Sensor Reflection)1. 一致性與數值差異：從散點圖可見，雖然兩者年均 NDVI 的線性相關性不高（$R^2 = 0.084$，推測受兩顆衛星過境時間、雲覆蓋率與合成取樣差異影響），但所有資料點均落在 1:1 對角線下方，明確顯示 Sentinel-2 的 NDVI 數值普遍低於 Landsat。這是因為 10m 的高解析度大幅降低了「混合像素 (Mixed Pixel)」效應，能精準獨立出林間陰影、狹窄破碎帶與裸地，從而拉低了整體的平均綠度。2. 損害面積估算的解析度效應 (Resolution Effect)：比較 2024 地震的 ΔNDVI 嚴重受損面積（<-0.15），Landsat 估算（約 7385 公頃）明顯高於 Sentinel-2（約 5737 公頃）。Landsat 的 30m 大像素容易將微小崩塌與周邊健康的樹冠混合，導致整個 900 平方公尺的像素數值被「平均」下拉並跌破閾值，進而高估了整體的受損範圍；而 Sentinel-2 (10m) 則能銳利地把真實崩塌與健康植被區隔開來，面積估算更為精確收斂。3. 方法論反思 (未來應用)：兩款感測器並非互相取代，而是高度互補。在未來的實務專案（如嘉義市淹水與交通路網分析）中，應優先使用 Landsat 建立 20 年以上的「宏觀環境基線與長期趨勢」；但在標定具體的「交通節點災損、微型中斷路段、精確淹水邊界」時，必須切換至 Sentinel-2，利用其 10m 解析度消除混合像素的誤差，精確圈繪出需工程介入的搶修熱區。